SECTION 1 - Install Dependencies



In [ ]:
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
!pip install ultralytics roboflow opencv-python-headless tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 56.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


SECTION 2 — Import Libraries

In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================
from ultralytics import YOLO
from roboflow import Roboflow
import os, cv2, shutil, random, yaml
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


SECTION 3 — Download Dataset (Daytime Baseline + Version 4 Nighttime Set)

In [ ]:
# ============================================================
# DOWNLOAD DATASETS FROM ROBOFLOW
# ============================================================
rf = Roboflow(api_key="UrwiWP3NYmsHgdFN4y39")

# Version 4 → Baseline (Daytime) - Using version 4 as version 1 was not found
project = rf.workspace("engg-680-project").project("pothole-detection-bfeeg-grxp2")
ds_day = project.version(4).download("yolov8")

# Version 4 → Used for Nighttime Augmentation
ds_night = project.version(4).download("yolov8")

print("Datasets downloaded successfully.")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Pothole-Detection-4 in yolov8:: 100%|██████████| 2272/2272 [00:00<00:00, 6558.10it/s]


Datasets downloaded successfully.


SECTION 4 — Train YOLOv8 (Baseline Daytime Model)

In [ ]:
# ============================================================
# BASELINE (DAYTIME) TRAINING USING YOLOv8s
# ============================================================

model_day = YOLO("yolov8s.pt")

results_day = model_day.train(
    data=f"{ds_day.location}/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    patience=10,
    name="baseline_daytime"
)

metrics_day = model_day.val(data=f"{ds_day.location}/data.yaml")

precision_day = metrics_day.box.p[0]
recall_day    = metrics_day.box.r[0]
f1_day        = metrics_day.box.f1[0]

print("=== BASELINE DAYTIME METRICS ===")
print(f"Precision: {precision_day:.3f}")
print(f"Recall:    {recall_day:.3f}")
print(f"F1 Score:  {f1_day:.3f}")


Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Pothole-Detection-4/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=baseline_daytime, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspective=0.0

SECTION 5 — Custom Nighttime Augmentation Pipeline

In [ ]:
# ============================================================
# CUSTOM NIGHTTIME AUGMENTATION FUNCTIONS
# ============================================================

def strong_darkening(image, factor=0.45):
    """Simulates nighttime by darkening, desaturating, and adding subtle blue tint."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 2] *= factor
    hsv[:, :, 1] *= 0.85
    hsv = np.clip(hsv, 0, 255).astype(np.uint8)
    img_dark = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    # Add subtle blue tint
    b, g, r = cv2.split(img_dark)
    b = cv2.addWeighted(b, 1.1, b, 0, 0)
    return cv2.merge([b, g, r])


def augment_dataset(images_folder, labels_folder, output_folder, transform, copies=1):
    """Applies augmentation + preserves label consistency."""
    os.makedirs(os.path.join(output_folder, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_folder, "labels"), exist_ok=True)

    for img_file in tqdm(os.listdir(images_folder)):
        if not img_file.lower().endswith((".jpg", ".png", ".jpeg")):
            continue

        img_path = os.path.join(images_folder, img_file)
        label = os.path.splitext(img_file)[0] + ".txt"
        label_path = os.path.join(labels_folder, label)

        if not os.path.exists(label_path):
            continue

        # Read original image
        img = cv2.imread(img_path)

        # Save original
        shutil.copy(img_path, os.path.join(output_folder, "images", img_file))
        shutil.copy(label_path, os.path.join(output_folder, "labels", label))

        # Save augmented copies
        for i in range(copies):
            aug = transform(img)
            new_img = f"{os.path.splitext(img_file)[0]}_aug{i}.jpg"
            new_lbl = f"{os.path.splitext(img_file)[0]}_aug{i}.txt"

            cv2.imwrite(os.path.join(output_folder, "images", new_img), aug)
            shutil.copy(label_path, os.path.join(output_folder, "labels", new_lbl))

SECTION 6 — Build Nighttime Dataset (Train + Val Augmented)

In [ ]:
# ============================================================
# APPLY NIGHTTIME AUGMENTATION TO TRAIN + VAL
# ============================================================

dataset_base = ds_night.location
train_img = os.path.join(dataset_base, "train/images")
train_lbl = os.path.join(dataset_base, "train/labels")
val_img   = os.path.join(dataset_base, "valid/images")
val_lbl   = os.path.join(dataset_base, "valid/labels")

output_base = "/content/nighttime_augmented"
train_out = os.path.join(output_base, "train")
val_out   = os.path.join(output_base, "val")

augment_dataset(train_img, train_lbl, train_out, strong_darkening, copies=1)
augment_dataset(val_img, val_lbl, val_out, strong_darkening, copies=1)

print("Nighttime augmentation complete.")


100%|██████████| 133/133 [00:00<00:00, 136.26it/s]

Nighttime augmentation complete.


SECTION 7 — Create New YAML for Nighttime Training

In [ ]:
# ============================================================
# CREATE UPDATED YAML FILE FOR NIGHTTIME TRAINING
# ============================================================

data_yaml = os.path.join(output_base, "data.yaml")

yaml_content = {
    'train': os.path.join(train_out, "images"),
    'val':   os.path.join(val_out, "images"),
    'nc': 1,
    'names': ['pothole']
}

with open(data_yaml, 'w') as f:
    yaml.dump(yaml_content, f)

print("Nighttime YAML created:", data_yaml)


Nighttime YAML created: /content/nighttime_augmented/data.yaml
